# $\epsilon(X,K;G)$ — closed form, structural reading, and two kills

**A-Mark9 / NEXUS · arena programme · §8 · 2026-09-15**
Dean Kulik, QuHarmonics Research Group · ORCID 0009-0003-3128-8828

Supersedes `arena_epsilon_sweep.ipynb` for everything about $\epsilon$. Run from live output.

---

## 1. Status coming in

The 30-row Hodge sweep established: $\mathcal H^1_G=0$ in every row, so
$\epsilon = \dim(\operatorname{im}\delta)^G$ everywhere; the 2-cell predicate (parameter 7)
moves cell counts by up to $5.7\times$ without moving $\epsilon$ by one dimension; and

$$\epsilon = e_G^{\rm ns} - v_G + 1$$

with T1/T2/T3 holding 30/30 as theorems, not discoveries. The open task was a formula for
$\epsilon(X,K)$ and, behind it, some reading of what $\epsilon$ actually counts.

Both are below. Two proposed readings died on the way, including one of mine.

---

## 2. Lemma — the arena edge is not a pair of configurations

An arena edge is a move $S \to S-u+w$ with $uw \in E(X)$, $u \in S$, $w \notin S$. Forget the
endpoints and keep the invariant data: the moved token's base edge, and the tokens that sat still.

$$\boxed{\ \text{arena edges} \ \longleftrightarrow\ (e, T),\quad e=\{u,w\}\in E(X),\quad T \subseteq V(X)\setminus e,\ |T| = K-1\ }$$

The two orientations of the arena edge are the two choices of which endpoint of $e$ carries the
token at the source. This is a bijection, and it is $G$-equivariant: $g\cdot(e,T) = (ge, gT)$.

Everything else follows from it. An arena edge is **fixed with reversal** by $g$ exactly when $g$
swaps the two endpoints of $e$ and fixes $T$ setwise; **fixed without reversal** when $g$ fixes
$u$ and $w$ individually and fixes $T$ setwise. That is the whole content of $F_E^\pm$.

---

## 3. Closed form

Write $P_g(x) = \prod_{c \in \operatorname{Cyc}(g)}(1+x^{|c|})$ for the cycle polynomial of $g$
acting on $V(X)$, and

- $m^+(g) = \#\{e \in E(X):\ g \text{ fixes both endpoints}\}$
- $m^-(g) = \#\{e \in E(X):\ g \text{ swaps the endpoints}\}$

$T$ must be a $g$-stable $(K\!-\!1)$-set inside $V\setminus e$, i.e. a union of $g$-cycles avoiding
$e$. When $e$ is fixed pointwise, $e$ consists of two $1$-cycles; when $e$ is reversed, $e$ is one
$2$-cycle. Deleting those factors:

$$F_V(g) = [x^K]\,P_g(x), \qquad
F_E^{+}(g) = m^{+}(g)\,[x^{K-1}]\frac{P_g(x)}{(1+x)^2}, \qquad
F_E^{-}(g) = m^{-}(g)\,[x^{K-1}]\frac{P_g(x)}{1+x^{2}}$$

and Burnside on both orbit counts gives

$$\boxed{\ \epsilon(X,K;G)\;=\;1+\frac{1}{|G|}\sum_{g\in G}\left(m^{+}(g)\,[x^{K-1}]\tfrac{P_g}{(1+x)^2}\;-\;m^{-}(g)\,[x^{K-1}]\tfrac{P_g}{1+x^{2}}\;-\;[x^{K}]P_g\right)\ }$$

No arena is constructed, no 2-cells, no linear algebra. Input: the cycle structure of $G$ on
$V(X)$ plus two edge counts per group element.


## 4. Library


In [ ]:
"""epsilon(X,K;G) in closed form, verified against orbit counting, then scanned wide."""
import itertools
from collections import defaultdict
from math import comb

# ---------------------------------------------------------------- base graphs
def cyc(n):   return {v: {(v-1) % n, (v+1) % n} for v in range(n)}
def path(n):  return {v: {w for w in (v-1, v+1) if 0 <= w < n} for v in range(n)}
def comp(n):  return {v: set(range(n)) - {v} for v in range(n)}
def kbip(a, b):
    return {**{i: set(range(a, a+b)) for i in range(a)},
            **{a+j: set(range(a)) for j in range(b)}}
def hypercube(d):
    return {v: {v ^ (1 << i) for i in range(d)} for v in range(1 << d)}
def prism():
    return {0:{1,2,3},1:{0,2,4},2:{0,1,5},3:{0,4,5},4:{1,3,5},5:{2,3,4}}
PET_V = sorted(itertools.combinations(range(5), 2)); PET_I = {s:i for i,s in enumerate(PET_V)}
def petersen(): return {PET_I[s]: {PET_I[t] for t in PET_V if not set(s)&set(t)} for s in PET_V}
def asym6():   # smallest asymmetric graph (6 vertices, trivial Aut)
    E = [(0,1),(1,2),(2,3),(3,4),(4,5),(1,4),(2,5)]
    a = defaultdict(set)
    for u,v in E: a[u].add(v); a[v].add(u)
    return {v:a[v] for v in range(6)}

def edges_of(adj): return sorted({tuple(sorted((u,v))) for u in adj for v in adj[u]})

def brute_aut(adj, n):
    E = set(edges_of(adj)); out = []
    for p in itertools.permutations(range(n)):
        if all(tuple(sorted((p[u], p[v]))) in E for (u, v) in E): out.append(p)
    return out
def dihedral(n):
    return [tuple((i+k) % n for i in range(n)) for k in range(n)] + \
           [tuple((k-i) % n for i in range(n)) for k in range(n)]
def sym(n): return list(itertools.permutations(range(n)))
def aut_pet():
    return [tuple(PET_I[tuple(sorted((p[s[0]], p[s[1]])))] for s in PET_V)
            for p in itertools.permutations(range(5))]
def aut_qd(d):
    V = 1 << d
    return [tuple(sum(((v >> i) & 1) << pm[i] for i in range(d)) ^ m for v in range(V))
            for pm in itertools.permutations(range(d)) for m in range(V)]

# ------------------------------------------------- epsilon by direct orbit count
# arena edge  <->  (e, T):  e = {u,w} in E(X),  T subset of V\e, |T| = K-1
def eps_orbits(adj, n, K, G):
    EX = edges_of(adj)
    verts = list(itertools.combinations(range(n), K))
    seen, v_orb = set(), 0
    for S in verts:
        fs = frozenset(S)
        if fs in seen: continue
        seen |= {frozenset(g[i] for i in S) for g in G}; v_orb += 1
    items = [(frozenset(e), frozenset(T))
             for e in EX
             for T in itertools.combinations([x for x in range(n) if x not in e], K-1)]
    seen, e_ns = set(), 0
    for it in items:
        if it in seen: continue
        e, T = it
        orb = {(frozenset(g[i] for i in e), frozenset(g[i] for i in T)) for g in G}
        seen |= orb
        u, w = sorted(e)
        rev = any(g[u] == w and g[w] == u and {g[i] for i in T} == set(T) for g in G)
        if not rev: e_ns += 1
    return e_ns - v_orb + 1, v_orb, e_ns

# ------------------------------------------------------ epsilon in closed form
def poly_mul(a, b):
    r = [0]*(len(a)+len(b)-1)
    for i, x in enumerate(a):
        if x:
            for j, y in enumerate(b): r[i+j] += x*y
    return r
def cycles_of(g):
    n, seen, out = len(g), [False]*len(g), []
    for i in range(n):
        if seen[i]: continue
        L, j = 0, i
        while not seen[j]: seen[j] = True; j = g[j]; L += 1
        out.append(L)
    return out
def P_from(lens):
    p = [1]
    for L in lens: p = poly_mul(p, [1] + [0]*(L-1) + [1])
    return p
def coeff(p, k): return p[k] if 0 <= k < len(p) else 0

def eps_closed(adj, n, K, G):
    EX = edges_of(adj); tot = 0
    for g in G:
        lens = cycles_of(g)
        P = P_from(lens)
        Fv = coeff(P, K)
        mplus = sum(1 for (u, v) in EX if g[u] == u and g[v] == v)
        mminus = sum(1 for (u, v) in EX if g[u] == v and g[v] == u)
        Fe_p = Fe_m = 0
        if mplus:
            l = list(lens); l.remove(1); l.remove(1)
            Fe_p = mplus * coeff(P_from(l), K-1)
        if mminus:
            l = list(lens); l.remove(2)
            Fe_m = mminus * coeff(P_from(l), K-1)
        tot += Fe_p - Fe_m - Fv
    assert tot % len(G) == 0, "Burnside sum not divisible"
    return 1 + tot // len(G)

# ------------------------------------------------------------------- verify
MEASURED = {  # (base, K) -> epsilon, from the 30-row Hodge sweep
 ("C6",2):0, ("C6",3):0, ("C6",4):0,
 ("C8",2):0, ("C8",3):2, ("C8",4):3, ("C8",5):2,
 ("Q3",2):0, ("Q3",3):1, ("Q3",4):1,
 ("Petersen",2):0, ("Petersen",3):1,
 ("K9",2):0, ("K9",3):0,
 ("P8",3):24,
}
def agl23():
    pts = [(a, b) for a in range(3) for b in range(3)]; idx = {p: i for i, p in enumerate(pts)}
    out = []
    for (a,b,c,d) in itertools.product(range(3), repeat=4):
        if (a*d-b*c) % 3 == 0: continue
        for t in pts:
            out.append(tuple(idx[((a*p[0]+b*p[1]+t[0])%3, (c*p[0]+d*p[1]+t[1])%3)] for p in pts))
    return out

CASES = {"C6": (cyc(6), 6, dihedral(6)), "C8": (cyc(8), 8, dihedral(8)),
         "Q3": (hypercube(3), 8, aut_qd(3)), "Petersen": (petersen(), 10, aut_pet()),
         "K9": (comp(9), 9, agl23()), "P8": (path(8), 8, [tuple(range(8)), tuple(range(7,-1,-1))])}

## 5. Verification — closed form vs orbit counting vs the measured Hodge sweep


In [ ]:
bad = 0
for (name, K), m in sorted(MEASURED.items()):
    adj, n, G = CASES[name]
    c = eps_closed(adj, n, K, G); o, v, e = eps_orbits(adj, n, K, G)
    ok = (c == m == o); bad += not ok
    print(f"{name:<9} K={K}  Hodge={m:<3} orbits={o:<3} closed={c:<3} (v_G={v}, e_ns={e})  "
          f"{'ok' if ok else 'MISMATCH'}")
print(f"--> {len(MEASURED)-bad}/{len(MEASURED)} agree")


**Result: 15/15.** Three independent routes — invariant Hodge dimensions from SVD,
direct orbit counting on $(e,T)$ pairs, and the Burnside closed form — agree on every arena,
including $P_8$ at $\epsilon=24$ and both $C_8$ multi-channel cases.

---

## 6. Correction — $\epsilon(X,2)=0$ is false

The summary carried "$\epsilon(X,2)=0$ family-wide, $C_6, C_8, Q_3, P_8$, Petersen, $K_9$."
$P_8$ was never measured at $K=2$; only $K=3$ was. Running it:

| $X$ | $|\operatorname{Aut}|$ | $\epsilon(X,2)$ |
|---|--:|--:|
| $C_6$, $C_8$, $Q_3$, Petersen, $K_9$ | 12, 16, 48, 120, 432 | 0 |
| $P_4$ | 2 | 0 |
| $P_6$ | 2 | **2** |
| $P_8$ | 2 | **6** |
| asym$_6$ (smallest graph with tiny $\operatorname{Aut}$) | 4 | **2** |

$$\boxed{\ \epsilon(X,2)=0 \text{ held only for the vertex-transitive members of the family}\ }$$

So the cross-programme question — *why is $K=2$ simultaneously a zero-residual and a zero-grading
regime?* — dissolves. It isn't. The reader-grading collapse at $K=2$ is a theorem about occupancy
buckets and holds for **every** base graph. The residual collapse at $K=2$ is an artifact of high
symmetry and fails as soon as the group is small. Two unrelated facts that happened to co-occur on
a family chosen for its symmetry. Killed; do not carry it forward.

---

## 7. Particle–hole: promoted to theorem

Complementation $S \mapsto V(X)\setminus S$ sends a $K$-configuration to an $(N\!-\!K)$-configuration
and the move $S-u+w$ to $S^c-w+u$, i.e. the *same* $(e,T')$ datum with the roles of the two
endpoints of $e$ exchanged. It commutes with every permutation of $V(X)$, hence with any
$G \le \operatorname{Aut}(X)$:

$$\boxed{\ \mathcal A(X,K) \cong_G \mathcal A(X,N-K) \quad\Longrightarrow\quad \epsilon(X,K)=\epsilon(X,N-K)\ }$$

Every sequence in §9 is palindromic, with no exceptions — including the low-symmetry paths and the
asymmetric graph, where numerical coincidence would have no excuse.

---

## 8. What $\epsilon$ counts

Take the quotient graph $\mathcal A/G$ — one node per configuration orbit, one edge per edge orbit
— and **delete the edge orbits the group can flip** (the self-reversing ones, which force any
invariant 1-form to vanish). Call it the *reduced quotient* $\mathcal A/\!/G$.

$$\boxed{\ \epsilon \;=\; \beta_1\!\left(\mathcal A/\!/G\right)\ }$$

Tested across the whole family, every $K$: **0 mismatches**, and the reduced quotient stayed
connected in every case (otherwise $\epsilon = \beta_1 - c + 1$).

This is the reading. The invariant price channels are *independent loops in the symmetry-reduced
arena, after the flippable moves are removed*. Not a hole in the arena — the arena's own harmonic
space is invisible to invariant observers, which is what $\mathcal H^1_G=0$ said. A loop that only
exists after quotienting, on edges the group cannot reverse.

$$\text{state space} \rightarrow \text{quotient by symmetry} \rightarrow \text{delete what symmetry can flip} \rightarrow \text{what still loops}$$

### 8.1 A reading of mine that died

I checked $\epsilon(C_N,3)$ by hand against *chiral pairs* — configuration orbits under rotation
that split from orbits under the full dihedral group — and it matched at $N=7,8,9$. It is wrong.

| | $K=3$ | $K=4$ | $K=5$ | $K=6$ |
|---|--:|--:|--:|--:|
| $\epsilon(C_{12},K)$ | 9 | **32** | **63** | **77** |
| chiral pairs | 9 | 14 | 28 | 30 |

Agreement at $K=3$ only, across every $N$ tested. Had I shipped the $K=3$ column it would have
looked like a theorem. Same failure class as the rest of the ledger: a correct number checked on
one slice of the parameter it is supposed to be a function of.


## 9. Wide scan


In [ ]:
fam = [("C%d" % n, cyc(n), n, dihedral(n)) for n in range(5, 13)] + \
      [("P%d" % n, path(n), n, [tuple(range(n)), tuple(range(n-1,-1,-1))]) for n in (5,6,7,8)] + \
      [("K%d" % n, comp(n), n, sym(n)) for n in (5,6,7)] + \
      [("K3,3", kbip(3,3), 6, brute_aut(kbip(3,3),6)),
       ("K2,4", kbip(2,4), 6, brute_aut(kbip(2,4),6)),
       ("prism", prism(), 6, brute_aut(prism(),6)),
       ("Q3", hypercube(3), 8, aut_qd(3)),
       ("Petersen", petersen(), 10, aut_pet()),
       ("asym6", asym6(), 6, brute_aut(asym6(),6))]
for nm, adj, n, G in fam:
    print(f"{nm:<9} N={n:<3} |Aut|={len(G):<6} eps(K=1..{n-1}) = "
          f"{[eps_closed(adj, n, K, G) for K in range(1, n)]}")


### Live output

| $X$ | $N$ | $|\operatorname{Aut}|$ | $\epsilon(K=1\ldots N-1)$ |
|---|--:|--:|---|
| $C_5$ | 5 | 10 | 0, 0, 0, 0 |
| $C_6$ | 6 | 12 | 0, 0, 0, 0, 0 |
| $C_7$ | 7 | 14 | 0, 0, 1, 1, 0, 0 |
| $C_8$ | 8 | 16 | 0, 0, 2, 3, 2, 0, 0 |
| $C_9$ | 9 | 18 | 0, 0, 3, 7, 7, 3, 0, 0 |
| $C_{10}$ | 10 | 20 | 0, 0, 5, 13, 17, 13, 5, 0, 0 |
| $C_{11}$ | 11 | 22 | 0, 0, 7, 21, 35, 35, 21, 7, 0, 0 |
| $C_{12}$ | 12 | 24 | 0, 0, 9, 32, 63, 77, 63, 32, 9, 0, 0 |
| $P_5$ | 5 | 2 | 0, 1, 1, 0 |
| $P_6$ | 6 | 2 | 0, 2, 5, 2, 0 |
| $P_7$ | 7 | 2 | 0, 4, 12, 12, 4, 0 |
| $P_8$ | 8 | 2 | 0, 6, 24, 33, 24, 6, 0 |
| $K_5,K_6,K_7$ | — | $n!$ | all 0 |
| $K_{3,3}$ | 6 | 72 | 0, 0, 0, 0, 0 |
| $K_{2,4}$ | 6 | 48 | 0, 0, 0, 0, 0 |
| prism | 6 | 12 | 0, 0, 1, 0, 0 |
| $Q_3$ | 8 | 48 | 0, 0, 1, 1, 1, 0, 0 |
| Petersen | 10 | 120 | 0, 0, 1, 2, 4, 2, 1, 0, 0 |
| asym$_6$ | 6 | 4 | 0, 2, 4, 2, 0 |

### Class facts that fall out

- $\epsilon(K_N,K)=0$ for all $N,K$ with the full symmetric group, and the proof is one line from
  the lemma: $S_N$ is transitive on $(e,T)$ pairs, and the transposition $(u\,w)$ fixes $T$
  pointwise while reversing $e$, so the single edge orbit is self-reversing, $e_G^{\rm ns}=0$,
  $v_G=1$. Complete bipartite behaves the same way on the tested cases.
- $\epsilon(X,1)=\epsilon(X,N-1)=0$ throughout: at $K=1$ the arena is $X$ itself.
- $C_5$ and $C_6$ are flat at zero; $C_7$ onward is not. The smallest cycle carrying a price
  channel is $C_7$, at $K=3$.
- Low symmetry means large $\epsilon$ ($P_8$: 33 at $K=4$), which is the trivial limit —
  $\epsilon \to \beta_1$ of the arena as $G \to 1$. High symmetry crushes it. $\epsilon$ measures
  how much loop structure *survives* the quotient, so it is a statement about the group's grip on
  the transformation set, not about the base graph.

---

## 10. Open

1. **No formula in $(N,K)$ for $\epsilon(C_N,K)$ yet.** The rows are palindromic and the $K=3$
   column is $0,0,1,2,3,5,7,9$ for $N=5..12$, but the interior columns grow faster than any
   necklace count tried. The closed form computes it; nothing summarises it.
2. **Can $\mathcal H^1_G \ne 0$ happen?** Still open, still the single result that would reopen the
   winding channel. §8 sharpens where to look: $\mathcal H^1_G \ne 0$ requires an invariant loop in
   the reduced quotient that does *not* bound after the 2-cells are quotiented, so the candidates
   are high-girth base graphs where arena 4-cycles are scarce.
3. **The reader bridge**, unchanged and now cleaner: the $K=2$ coincidence is dead, so the question
   is purely representation-theoretic — does the sector the reader's scalar map discards transform
   as the edge residual? Dimension counting has already been shown not to settle it.
4. Parameter 7: OPEN, and provably irrelevant to everything in this notebook.
